In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 lxml==5.3.0 openpyxl==3.1.5 rapidfuzz==3.9.7 tqdm==4.66.5

In [ ]:
import os,re,time,random,hashlib,unicodedata
from datetime import datetime, timezone
import pandas as pd
from rapidfuzz import fuzz

def remove_accents(text):
    text='' if text is None else str(text)
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
def normalize_text(text):
    t=remove_accents(text).lower(); t=re.sub(r'[^a-z0-9\s]',' ',t)
    return re.sub(r'\s+',' ',t).strip()
def normalize_name(name): return normalize_text(name)
def stable_hash(text): return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:16]
def score_name_match(a,b): return float(fuzz.token_sort_ratio(normalize_name(a),normalize_name(b)))
def classify_match(score):
    return 'exact_or_very_strong_name' if score>=95 else 'strong_candidate' if score>=88 else 'medium_candidate' if score>=75 else 'weak_candidate' if score>=60 else 'discard'
def now_utc(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path):
    if not path or not os.path.exists(path): return ''
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for ch in iter(lambda:f.read(8192),b''): h.update(ch)
    return h.hexdigest()
def save_csv(df,path): os.makedirs(os.path.dirname(path),exist_ok=True); df.to_csv(path,index=False,encoding='utf-8-sig')
EVID_COLS=['seed_id', 'nombre_persona_input', 'normalized_name_input', 'estado_input', 'puesto_input', 'dependencia_input', 'notas_input', 'source', 'source_type', 'source_country', 'source_official', 'source_category', 'source_reliability', 'matched_record_name', 'matched_record_normalized_name', 'matched_alias', 'matched_entity_type', 'matched_role', 'matched_position', 'matched_agency', 'matched_state', 'matched_country', 'matched_identifier', 'matched_company', 'match_score_pct', 'match_strength', 'match_method', 'matched_fields', 'conflicting_fields', 'requires_review', 'identity_confidence_pct', 'signal_type', 'signal_category', 'severity', 'risk_layer', 'risk_level_candidate', 'confidence_pct', 'evidence_title', 'evidence_summary', 'evidence_snippet', 'evidence_keywords', 'evidence_date', 'evidence_url', 'source_url', 'search_query', 'search_engine', 'retrieved_at', 'raw_file_path', 'raw_file_sha256', 'involvement_summary', 'explanation', 'limitations', 'recommended_action']

In [ ]:

import glob
FORCE_DOWNLOAD=False
ENABLE_LARGE_DOWNLOADS=False
peps=pd.read_csv('notebooks/output/00_peps_normalized.csv',dtype=str).fillna('')
folders={'SAT_69B':'sat_69b','SERVIDORES_SANCIONADOS':'servidores_sancionados','COMPRANET':'compranet','DECLARANET':'declaranet'}
for f in folders.values(): os.makedirs(f'notebooks/raw/{f}',exist_ok=True)
bench=[]; all_rows=[]
def process_source(src,folder,signal):
    files=glob.glob(f'notebooks/raw/{folder}/*')
    rows=[]; err=''
    if not files: err='No local files found and automatic download disabled/unavailable.'
    for fp in files:
        try:
            d=pd.read_csv(fp,dtype=str).fillna('') if fp.lower().endswith('.csv') else pd.read_excel(fp,dtype=str).fillna('')
            name_col=next((c for c in d.columns if 'nombre' in normalize_name(c) or 'razon' in normalize_name(c)), d.columns[0])
            for _,p in peps.iterrows():
                for nm in d[name_col].astype(str).head(3000):
                    sc=score_name_match(p['nombre_persona_input'],nm)
                    if classify_match(sc)!='discard':
                        rows.append({**{c:'' for c in EVID_COLS},'seed_id':p['seed_id'],'nombre_persona_input':p['nombre_persona_input'],'normalized_name_input':p['normalized_name_input'],'estado_input':p.get('estado_input',''),'puesto_input':p.get('puesto_input',''),'dependencia_input':p.get('dependencia_input',''),'notas_input':p.get('notas_input',''),'source':src,'source_type':'mex_public_registry','source_country':'MX','source_official':True,'source_category':'public_records','source_reliability':'official_or_public','matched_record_name':nm,'matched_record_normalized_name':normalize_name(nm),'match_score_pct':sc,'match_strength':classify_match(sc),'requires_review':True,'identity_confidence_pct':min(sc,75),'signal_type':signal,'signal_category':'mex_public_source','severity':'medium','risk_layer':'public_records','risk_level_candidate':'medium_review','confidence_pct':min(sc,75),'retrieved_at':now_utc(),'raw_file_path':fp,'raw_file_sha256':sha256_file(fp),'involvement_summary':'Registro público asociado a nombre similar; no se confirma identidad solo con nombre.','recommended_action':'verify_identity'})
        except Exception as e:
            err=str(e)
    bench.append({'source':src,'source_type':'mex_public_registry','attempted':True,'success':len(rows)>0,'rows_downloaded_or_loaded':len(files),'rows_parsed':len(rows),'matches_found':len(rows),'unique_people_with_hits':len(set([r['seed_id'] for r in rows])),'runtime_seconds':0,'error_message':err if err else ('No local files found and automatic download disabled/unavailable.' if not files else ''),'notes':'Manual file mode'})
    return rows
parts={
'02_sat_69b_evidence.csv':process_source('SAT_69B','sat_69b','sat_69b_candidate'),
'02_sancionados_evidence.csv':process_source('SERVIDORES_SANCIONADOS','servidores_sancionados','servidor_publico_sancionado_candidate'),
'02_compranet_evidence.csv':process_source('COMPRANET','compranet','contrato_publico_candidate'),
'02_declaranet_evidence.csv':process_source('DECLARANET','declaranet','declaranet_record_candidate')}
for f,r in parts.items(): save_csv(pd.DataFrame(r,columns=EVID_COLS),f'notebooks/output/{f}'); all_rows.extend(r)
ev=pd.DataFrame(all_rows,columns=EVID_COLS); save_csv(ev,'notebooks/output/02_mexico_fuentes_publicas_evidence.csv')
ps=ev.groupby(['seed_id','nombre_persona_input'],as_index=False).size().rename(columns={'size':'total_signals'}) if len(ev) else pd.DataFrame(columns=['seed_id','nombre_persona_input','total_signals'])
save_csv(ps,'notebooks/output/02_mexico_fuentes_publicas_person_summary.csv'); save_csv(pd.DataFrame(bench),'notebooks/output/02_benchmark_mexico_fuentes_publicas.csv')
